# Train the XGBoost scorer (offline, on the LinuxONE VM)

Runs inside the LinuxONE Jupyter container. Reads Pair 1's cleaned table from `/home/ibm-user/shared/data/`
(the VM's `~/shared/data`) and writes the feature table and the model files everyone else reads.

Sections:
1. Load the clean table and the time split (made by `scripts/clean_data.py`)
2. Clues (features), past-only, through the shared `common/features.py`
3. Train XGBoost with early stopping, three seeds
4. Pick thresholds on validation
5. Evaluate on test, once: each seed, the seeds averaged, split at September 11, precision among the top flags
6. Sanity checks: clue averages and the leakage test

Sections 3 to 5 use the same functions as `scripts/train_xgboost.py`, so the notebook and the script give the same numbers.
Takes about 15 to 30 minutes end to end on the VM. The kernel runs inside the container, so closing the browser or losing SSH does not stop it.
Each section saves its output, so you can restart from any section by loading the saved file.

In [ ]:
import os, sys, json, time
from pathlib import Path

import numpy as np
import pandas as pd

# the repo root, so `common` imports; the notebook lives in <repo>/notebooks/
REPO = Path.cwd().resolve()
while not (REPO / "common").exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
from common.clean_transactions import read_clean_csv
from common.features import ClueBuilder, FEATURE_NAMES

DATA = Path(os.environ.get("DATA_DIR", "/home/ibm-user/shared/data"))
MODELS = Path(os.environ.get("MODELS_DIR", "/home/ibm-user/shared/models"))
MODELS.mkdir(parents=True, exist_ok=True)

# CSV everywhere: the LinuxONE image has no pyarrow, so no parquet
def save_csv(df, name):
    df.to_csv(DATA / f"{name}.csv", index=False)

## 1. Load the clean table and the time split

Pair 1's cleaning script writes `clean.csv` and `splits.json`. The table is already sorted by time,
has `txn_id`, the `<bank>_<account>` IDs, `amount_usd`, the label, and a `split` column (train / validation / test by time).
Nothing is recomputed here; every pair uses the same file.

In [ ]:
t0 = time.time()
# only the columns this notebook uses: the VM has 8 GB and no pyarrow, so text columns are costly
COLUMNS = ["txn_id", "ts", "from_acct", "to_acct", "amt_paid", "ccy_paid", "amt_recv", "ccy_recv",
           "amount_usd", "fmt", "label", "split"]
df = read_clean_csv(DATA / "clean.csv", columns=COLUMNS)   # restores the column types the CSV loses
splits = json.load(open(DATA / "splits.json"))
print(f"{len(df):,} rows loaded in {time.time() - t0:.0f}s")
print("date range:", df.ts.min(), "to", df.ts.max())
print("laundering rows:", int(df.label.sum()), f"({df.label.mean():.4%})")
print("validation starts:", splits["validation_start"], "| test starts:", splits["test_start"])
df.groupby("split", observed=True).label.agg(rows="size", laundering="sum", rate="mean")

## 2. Clues

Every row goes through `ClueBuilder` in time order. The class computes the clues for a transfer *before* adding it to the
accounts' memory, so nothing from the future leaks in. The same class runs inside the live scorer.

Expect a few minutes for 5 million rows; progress prints every 500,000.

In [ ]:
t0 = time.time()
n = len(df)
builder = ClueBuilder()
X = np.empty((n, len(FEATURE_NAMES)), dtype=np.float32)

ts_min = df.ts.values.astype("datetime64[m]").astype("int64")
cols = zip(ts_min, df.from_acct.astype(str).values, df.to_acct.astype(str).values, df.amount_usd.values,
           df.ccy_paid.astype(str).values, df.ccy_recv.astype(str).values, df.fmt.astype(str).values)
for i, (t, fa, ta, amt, cp, cr, f) in enumerate(cols):
    X[i] = builder.compute_row(t, fa, ta, amt, cp, cr, f)
    if i % 500_000 == 0 and i:
        print(f"{i:,} rows, {time.time() - t0:.0f}s")

feat = pd.DataFrame(X, columns=FEATURE_NAMES)
feat.insert(0, "txn_id", df.txn_id.values)
feat.insert(1, "split", df.split.astype(str).values)
feat["amount_usd"] = df.amount_usd.values   # for the baseline only, never a feature
feat["label"] = df.label.values
assert not feat[FEATURE_NAMES].isna().any().any(), "NaN in clues"
save_csv(feat, "features")
del builder, X, cols   # free the per-account memory; the leakage test builds its own
import gc; gc.collect()
print(f"done: {len(feat):,} rows x {len(FEATURE_NAMES)} clues in {time.time() - t0:.0f}s")

## 3. Train XGBoost, three seeds

`scale_pos_weight` makes each laundering row count as much as all the normal rows together.
Early stopping on validation PR-AUC picks the number of trees, and every score below uses only the trees up to that
best iteration, exactly as the live scorer does. Results swing between seeds, so three are trained and their
probabilities averaged. The scorer loads one model, so the seed with the best validation PR-AUC is the one saved.

In [ ]:
import xgboost as xgb
from scripts.train_xgboost import (BASE_PARAMS, SPLIT_DATE, choose_thresholds, format_table, predict,
                                   split_tables, train_config)

if "feat" not in globals():
    feat = pd.read_csv(DATA / "features.csv", dtype={name: "float32" for name in FEATURE_NAMES})
split = feat.split.to_numpy()
y_all = feat.label.to_numpy().astype(int)
X_train, y_train = feat.loc[split == "train", FEATURE_NAMES].to_numpy(np.float32), y_all[split == "train"]
X_val, y_val = feat.loc[split == "validation", FEATURE_NAMES].to_numpy(np.float32), y_all[split == "validation"]
X_test, y_test = feat.loc[split == "test", FEATURE_NAMES].to_numpy(np.float32), y_all[split == "test"]
test = feat[split == "test"]
ts_test = df.ts.values[split == "test"]          # feat and df are in the same row order
print("xgboost", xgb.__version__, "| train", len(y_train), "validation", len(y_val), "test", len(y_test))

In [ ]:
SEEDS = [0, 1, 2]
t0 = time.time()
run = train_config("base", BASE_PARAMS, X_train, y_train, X_val, y_val, FEATURE_NAMES, SEEDS, os.cpu_count())
del X_train
print(f"trained {len(SEEDS)} seeds in {time.time() - t0:.0f}s, best iterations {run['best_iteration']}")
print("validation PR-AUC per seed:", [round(a, 4) for a in run["val_ap"]],
      f"| averaged: {run['val_ap_avg']:.4f} (laundering rate {y_val.mean():.4f})")

best = int(np.argmax(run["val_ap"]))
booster = run["boosters"][best]
booster.save_model(MODELS / "scorer.json")
json.dump(FEATURE_NAMES, open(MODELS / "feature_list.json", "w"), indent=2)
print("saved seed", SEEDS[best])

## 4. Thresholds, chosen on validation only

- **alert**: the score above which one transfer is flagged. Best F1 on validation.
- **ring**: the score above which a transfer may join a ring. Lowest score with validation precision of at least 0.25.
  Lower than alert on purpose: rings need more edges.

Each seed and the average get their own thresholds; `thresholds.json` holds the saved seed's.

In [ ]:
seed_thresholds = [choose_thresholds(y_val, p) for p in run["p_val"]]
avg_thresholds = choose_thresholds(y_val, run["p_val_avg"])
thresholds = dict(seed_thresholds[best], seed=SEEDS[best], config="base")
json.dump(thresholds, open(MODELS / "thresholds.json", "w"), indent=2)
pd.DataFrame([*seed_thresholds, avg_thresholds],
             index=[f"seed {s}" for s in SEEDS] + ["average"]).drop(columns="chosen_on").round(4)

## 5. Evaluate on test, once

Precision, recall, F1 and the confusion counts are at each row's validation-chosen alert threshold; PR-AUC is
average precision; `p@K` is the share of laundering among the K highest scores. At a 0.1 % laundering rate
accuracy is meaningless, so it is not shown. The baseline ("flag everything above $10,000") shows what the model adds.

About a third of the test laundering falls after September 10, when normal traffic nearly stops; that tail
flatters every score, so the table is also split at September 11.

In [ ]:
p_seeds = [predict(b, X_test, FEATURE_NAMES) for b in run["boosters"]]
p_test = np.mean(p_seeds, axis=0)
entries = [(f"seed {s}", p, t["alert"]) for s, p, t in zip(SEEDS, p_seeds, seed_thresholds)]
entries.append((f"average of {len(SEEDS)} seeds", p_test, avg_thresholds["alert"]))
tables = split_tables(entries, y_test, ts_test, test.amount_usd.to_numpy())
print(f"test rows {len(y_test):,}, laundering {int(y_test.sum()):,} ({y_test.mean():.4%})")
format_table(tables["overall"])

In [ ]:
print(f"before {SPLIT_DATE[:10]}")
display(format_table(tables["before"]))
print(f"from {SPLIT_DATE[:10]} on")
format_table(tables["after"])

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve

pt, rt, _ = precision_recall_curve(y_test, p_test)
plt.figure(figsize=(5, 4))
plt.plot(rt, pt, label=f"XGBoost, {len(SEEDS)} seeds averaged (PR-AUC {tables['overall'].loc[entries[-1][0], 'pr_auc']:.3f})")
plt.axhline(y_test.mean(), color="grey", ls="--", label=f"laundering rate {y_test.mean():.4f}")
plt.xlabel("recall"); plt.ylabel("precision"); plt.title("Test precision-recall"); plt.legend()
plt.tight_layout(); plt.show()

In [ ]:
# which clues drive the saved model's score (gain). pass-through and new-payee clues should rank high;
# if something odd like hour is on top, check for leakage.
gain = booster.get_score(importance_type="gain")
imp = pd.Series(gain, name="gain").reindex(FEATURE_NAMES).fillna(0).sort_values(ascending=False)
imp.round(1).head(15)

In [ ]:
# the three reasons: SHAP contributions of the saved model for the highest-scoring test transfers
p_saved = p_seeds[best]
top = np.argsort(-p_saved)[:5]
contrib = booster.predict(xgb.DMatrix(X_test[top], feature_names=FEATURE_NAMES), pred_contribs=True,
                          iteration_range=(0, booster.best_iteration + 1))[:, :-1]   # last column is bias
for k, i in enumerate(top):
    order = np.argsort(-contrib[k])[:3]
    reasons = [f"{FEATURE_NAMES[j]}={X_test[i, j]:.2f} (+{contrib[k, j]:.2f})" for j in order]
    print(f"txn {int(test.txn_id.iloc[i])}  score {p_saved[i]:.3f}  label {int(y_test[i])}  ->  {', '.join(reasons)}")

## 6. Sanity checks

**Clue averages**: each clue should differ between laundering and normal rows in a sensible direction.
This table is a pitch slide.

**Leakage test**: push the first 1,000 transfers through a fresh `ClueBuilder` one at a time using the live `compute()` path
(text fields, exactly as they arrive from Redis) and compare with the offline table. They must match exactly.

In [ ]:
means = feat.groupby("label")[FEATURE_NAMES].mean().T
means.columns = ["normal", "laundering"]
means["ratio"] = means.laundering / means.normal.replace(0, np.nan)
means.round(3)

In [ ]:
head = df.head(1000)
live = ClueBuilder()
rows = []
for r in head.itertuples(index=False):
    msg = {  # exactly the txns-stream fields, all text
        "txn_id": str(r.txn_id), "ts": pd.Timestamp(r.ts).isoformat(),
        "from_acct": str(r.from_acct), "to_acct": str(r.to_acct), "amount_usd": str(r.amount_usd),
        "amt_paid": str(r.amt_paid), "ccy_paid": str(r.ccy_paid),
        "amt_recv": str(r.amt_recv), "ccy_recv": str(r.ccy_recv), "fmt": str(r.fmt),
    }
    rows.append(live.compute(msg))
live_X = pd.DataFrame(rows)[FEATURE_NAMES].to_numpy(dtype=np.float32)
offline_X = feat[FEATURE_NAMES].head(1000).to_numpy(dtype=np.float32)
mismatch = np.abs(live_X - offline_X).max()
print("max difference live vs offline:", mismatch)
assert mismatch == 0, "live and offline clues differ: train/serve skew"
print("leakage test passed")

## Files written

| File | Used by |
|------|---------|
| `data/features.csv` | GNN training |
| `models/scorer.json` | scorer (the seed with the best validation PR-AUC) |
| `models/feature_list.json` | scorer, rings |
| `models/thresholds.json` | scorer, rings |

Inputs `data/clean.csv` and `data/splits.json` come from Pair 1's `scripts/clean_data.py`.
Numbers for the pitch: the test tables in section 5 (overall and split at September 11), the validation PR-AUC in
section 3, and the clue averages in section 6. `scripts/train_xgboost.py` prints the same tables from the command line.